In [1]:
%pwd

'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project\\research'

In [2]:
# we want to go to root directory
import os

os.chdir("../")

In [14]:
%pwd

'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project'

**STEP 1-3 FROM README , WE CREATE CONFIG FILES**

**STEP 4 CREATE THE ENTITY**

In [15]:
# STEP 1-3 FROM README , WE CREATE CONFIG FILES

# we want to make an ENTITY (Class) for our data injestion OUTPUT coming feorm google drive url

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [16]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

**STEP 5 : CREATE configuration manager**

In [24]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # create the root artifacts folder
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
            
            # get config under data ingestion
        config = self.config.data_ingestion
            
            # create the root directory under data ingestion folder
        create_directories([config.root_dir])

            # get the data ingestion config as an object
        data_ingestion_config = DataIngestionConfig(
                root_dir=config.root_dir,
                source_URL=config.source_URL,
                local_data_file=config.local_data_file,
                unzip_dir=config.unzip_dir 
            )

        return data_ingestion_config    

**STEP 6 : CREATE COMPONENT**

In [25]:
import os
import zipfile
import gdown
from cnnClassifier import logger
# from cnnClassifier.utils.common import get_size

In [26]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            #file_id = dataset_url.split("/")[-2]
            #prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(dataset_url,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

**STEP 7: PIPELINE**

In [27]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-09-21 20:07:41,268: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-09-21 20:07:41,270: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-09-21 20:07:41,271: INFO: common: Created directory at: artifacts]
[2026-09-21 20:07:41,273: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-09-21 20:07:41,276: INFO: 137279635: Downloading data from https://drive.google.com/file/d/1N927NnbsuBOOBmK4VrTnA1Ya04QB1G3t/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1N927NnbsuBOOBmK4VrTnA1Ya04QB1G3t
From (redirected): https://drive.google.com/uc?id=1N927NnbsuBOOBmK4VrTnA1Ya04QB1G3t&confirm=t&uuid=b72eaee9-8e35-470b-92c5-fa8cab568c5a
To: d:\Projects\1-Practice and Learning\0 - COMPLETE NEW LEARNING ML and DL\3-PROJECTS\Deep Learning Projects\Chest Cancer Classification End to End Project\artifacts\data_ingestion\data.zip
100%|██████████| 80.5M/80.5M [01:10<00:00, 1.15MB/s]

[2026-09-21 20:08:55,769: INFO: 137279635: Downloaded data from https://drive.google.com/file/d/1N927NnbsuBOOBmK4VrTnA1Ya04QB1G3t/view?usp=sharing into file artifacts/data_ingestion/data.zip]


**EXTRA-STEP STEP 8 : MOVE THIS ENTIRE PIPELINE TO A .PY FILE**